In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import pandas as pd  # 添加缺失的pandas导入

# 文件路径
MERRA2_FILE = "/mnt/backup_ETH/Marina/MERRA2/daily_fields/MERRA2_3d_dm_r144x96_1980-05.2020_SLP_O3.nc"
output_dir = '/home/weiji/restart_exam/plots/MERRA2_O3/'
save_path = os.path.join(output_dir, "MERRA2_O3_Anomaly_2020.png")
save_path_clim = os.path.join(output_dir, "MERRA2_O3_Climatology.png")
save_path_clim_ppm = os.path.join(output_dir, "MERRA2_O3_Climatology_ppm.png")
save_path_anom_ppm = os.path.join(output_dir, "MERRA2_O3_Anomaly_2020_ppm.png")

# 定义单位转换因子：kg/kg -> mol/mol
M_air = 28.97  # 空气摩尔质量 (g/mol)
M_O3 = 48.0    # 臭氧摩尔质量 (g/mol)
conversion_factor = M_air / M_O3  # 约 0.6035

# 处理 O3 数据：经度平均和纬度加权平均 (60-90°N)
def process_o3(da):
    da = da.mean(dim='lon')  # 经度平均
    da = da.sel(lat=slice(60, 90))  # 选取 60-90°N
    weights = np.cos(np.deg2rad(da.lat))  # 计算纬度的余弦权重
    da = da.weighted(weights).mean(dim='lat')  # 纬度加权平均
    return da

# 加载数据
ds = xr.open_dataset(MERRA2_FILE)
# 处理 O3 数据
da = ds['O3']
# 计算气候态 (1980-2019 年 1-5 月)
da_clim_full = da.sel(time=da.time.dt.month.isin(range(1, 6)))
da_clim_full = da_clim_full.sel(time=da_clim_full.time.dt.year.isin(range(1980, 2020)))
da_clim = da_clim_full.groupby('time.dayofyear').mean(dim='time')
da_clim = process_o3(da_clim)

# 提取 2020 年数据
da_2020 = da.sel(time=da.time.dt.year == 2020)
da_2020 = da_2020.sel(time=da_2020.time.dt.month.isin(range(1, 6)))
da_2020 = process_o3(da_2020)

# 计算异常值
time_vals_2020 = da_2020['time'].values
dayofyear_2020 = da_2020['time'].dt.dayofyear.values
da_clim_aligned = da_clim.sel(dayofyear=dayofyear_2020)
da_clim_aligned = da_clim_aligned.assign_coords(time=('dayofyear', time_vals_2020))
da_clim_aligned = da_clim_aligned.swap_dims({'dayofyear': 'time'})
anom_da = da_2020 - da_clim_aligned

# 准备绘图数据（针对 anom_da）
da = anom_da
da = da.sel(time=da.time.dt.month.isin(range(1, 6)))
da = da.transpose('lev', 'time')
time_vals = da['time'].values
lev_vals = da['lev'].values  # 原单位为 hPa
anom_array = da.values
anom_array_kg = anom_array  # kg/kg 单位异常值
# 对于 mol/mol 图先转换成mol/mol，再乘以1e6得到 ppm
anom_array_ppm = anom_array * conversion_factor * 1e6  # ppm 单位异常值

# 准备绘图数据（针对 clim_da）
da = da_clim_aligned
da = da.sel(time=da.time.dt.month.isin(range(1, 6)))
da = da.transpose('lev', 'time')
clim_time_vals = da['time'].values
clim_lev_vals = da['lev'].values  # 单位为 hPa
clim_array = da.values
clim_array_kg = clim_array  # kg/kg 单位气候态
clim_array_ppm = clim_array * conversion_factor * 1e6  # ppm 单位气候态

# 绘图函数（异常值）
def plot_merra2_o3_anomaly(anom_da, clim_da, save_path, unit='kg/kg'):
    fig, ax = plt.subplots(figsize=(6, 4))
    time_mpl = mdates.date2num(anom_da.time.values)
    # 将层高从 hPa 转换为 Pa
    lev_vals = anom_da.lev.values * 100

    # 当单位为 ppm 时，固定色标范围应为
    if unit == 'ppm':
        contour_levels = np.linspace(-1.5, 1.5, 31)  #
    else:
        contour_levels = 20  # kg/kg 时采用默认分级数

    cf = ax.contourf(time_mpl, lev_vals, anom_da.values, levels=contour_levels, cmap='RdBu_r')
    
    clim_data = clim_da.values
    if unit == 'kg/kg':
        # 对应 kg/kg 阈值（原来以mol/mol给出的阈值转换为kg/kg）
        clim_levels = [x / conversion_factor for x in [4.8e-6, 5.2e-6, 5.6e-6, 6.0e-6, 6.4e-6]]
    elif unit == 'ppm':
        # 对应 ppm 阈值：原先的阈值 [4.8e-6,5.2e-6,...] 乘以 1e6 得到 [4.8,5.2,...]
        clim_levels = [4.8, 5.2, 5.6, 6.0, 6.4]
    else:
        clim_levels = [4.8e-6, 5.2e-6, 5.6e-6, 6.0e-6, 6.4e-6]
        
    CS = ax.contour(time_mpl, lev_vals, clim_data, levels=clim_levels, colors='k', linewidths=1.5)
    # 根据单位选择合适的标签格式：ppm时不使用科学计数法
    fmt = '%.1e'
    if unit == 'ppm':
        fmt = '%.1f'
    ax.clabel(CS, inline=True, fontsize=8, fmt=fmt)

    ax.set_yscale('log')
    ax.invert_yaxis()
    ax.set_xlim(time_mpl[0], time_mpl[-1])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_ylabel('Pressure (Pa)', fontsize=10)
    ax.set_title('MERRA2 O3 Anomaly (2020) with Climatology Contours', fontsize=12)

    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(cf, cax=cax, label=f'O3 Anomaly ({unit})')

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(fig)

# 绘制异常图
anom_plot_da_kg = xr.DataArray(anom_array_kg, coords={'lev': lev_vals, 'time': time_vals}, dims=['lev', 'time'])
clim_plot_da_kg = xr.DataArray(clim_array_kg, coords={'lev': clim_lev_vals, 'time': clim_time_vals}, dims=['lev', 'time'])
anom_plot_da_ppm = xr.DataArray(anom_array_ppm, coords={'lev': lev_vals, 'time': time_vals}, dims=['lev', 'time'])
clim_plot_da_ppm = xr.DataArray(clim_array_ppm, coords={'lev': clim_lev_vals, 'time': clim_time_vals}, dims=['lev', 'time'])

plot_merra2_o3_anomaly(anom_plot_da_kg, clim_plot_da_kg, save_path, unit='kg/kg')
plot_merra2_o3_anomaly(anom_plot_da_ppm, clim_plot_da_ppm, save_path_anom_ppm, unit='ppm')

# 为 da_clim 添加时间坐标
dates_2020 = pd.date_range(start='2020-01-01', end='2020-05-31', freq='D')
dayofyear_2020 = dates_2020.dayofyear
da_clim_aligned = da_clim.sel(dayofyear=dayofyear_2020)
da_clim_aligned = da_clim_aligned.assign_coords(time=('dayofyear', dates_2020))
da_clim_aligned = da_clim_aligned.swap_dims({'dayofyear': 'time'})

# 准备绘图数据（气候态）
da = da_clim_aligned
da = da.transpose('lev', 'time')
time_vals = da['time'].values
lev_vals = da['lev'].values  # 单位为 hPa
clim_array = da.values
clim_array_kg = clim_array  # kg/kg
clim_array_ppm = clim_array * conversion_factor * 1e6  # ppm

# 绘制 O3 气候态垂直剖面图
def plot_o3_climatology(clim_da, save_path, unit='kg/kg'):
    fig, ax = plt.subplots(figsize=(6, 4))
    time_mpl = mdates.date2num(clim_da.time.values)
    # 将层高从 hPa 转换为 Pa
    lev_vals = clim_da.lev.values * 100

    cf = ax.contourf(time_mpl, lev_vals, clim_da.values, levels=20, cmap='viridis')

    ax.set_yscale('log')
    ax.invert_yaxis()
    ax.set_xlim(time_mpl[0], time_mpl[-1])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_ylabel('Pressure (Pa)', fontsize=10)
    ax.set_title('MERRA2 O3 Climatology (1980-2019)', fontsize=12)

    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(cf, cax=cax, label=f'O3 ({unit})')

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(fig)

# 绘制气候态图
clim_plot_da_kg = xr.DataArray(clim_array_kg, coords={'lev': lev_vals, 'time': time_vals}, dims=['lev', 'time'])
clim_plot_da_ppm = xr.DataArray(clim_array_ppm, coords={'lev': lev_vals, 'time': time_vals}, dims=['lev', 'time'])

plot_o3_climatology(clim_plot_da_kg, save_path_clim, unit='kg/kg')
plot_o3_climatology(clim_plot_da_ppm, save_path_clim_ppm, unit='ppm')
